In [1]:
import json
import os
import uuid
from abc import ABC, abstractmethod
from enum import Enum
from pathlib import Path
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

In [2]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

In [3]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

In [4]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [5]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [6]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [7]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

In [8]:
def message_from_dict(data: dict) -> Message:
    classes = {
        MsgType.SYSTEM.value: SystemMessage,
        MsgType.USER.value: UserMessage,
        MsgType.ASSISTANT.value: AssistantMessage,
    }
    return classes[data["type"]](data["content"])

In [9]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

In [10]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, message: Message):
        self.messages.append(message)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

In [11]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

In [12]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

In [13]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

In [14]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

In [15]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage):
        self.provider = provider
        self.system = system
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    note(f"loaded memory #{memory.id}", memory.history())
                continue

            user_message = UserMessage(message)
            response = self.provider.call(system or self.system, self.memory.messages + [user_message])
            self.memory.add(user_message)
            self.memory.add(response)
            self.store.save(self.memory)
            print(f"AI: {response.summary()}")

In [16]:
provider = OpenAIProvider()
agent = Agent(provider, system_message)
agent.chat(["你好，我叫 Steven，正在学第 2 部分。"])
saved_id = agent.memory.id  # this memory is saved in memories/

  * memory: #43145392

YOU: 你好，我叫 Steven，正在学第 2 部分。
AI: 你好 Steven，很高兴一起学习《从大模型到智能体》第二部分！

我可以帮你做这些事：
- 用更容易懂的话解释章节内容
- 帮你梳理知识框架
- 结合代码讲 Agent 的核心概念
- 帮你总结重点、做题、答疑
- 设计小练习，带你一步步理解

如果你愿意，可以直接告诉我：
1. 你正在看第几章/哪一节  
2. 哪个概念不太懂  
3. 你想要“通俗解释”还是“结合代码讲解”

你把内容发给我，我们就可以开始。

YOU: /quit


In [17]:
# start a new memory, then load the saved one again
agent.chat(["/new", "我叫什么名字？", f"#{saved_id}", "我叫什么名字？"])
# or call agent.chat() without questions to type your own

  * memory: #43145392

YOU: /new
  * new memory: #d74106aa

YOU: 我叫什么名字？
AI: 我目前不知道你的名字。  
如果你愿意，可以告诉我，我就记住并称呼你。

YOU: #43145392
  * loaded memory #43145392: [User]: 你好，我叫 Steven，正在学第 2 部分。
    [Assistant]: 你好 Steven，很高兴一起学习《从大模型到智能体》第二部分！
    
    我可以帮你做这些事：
    - 用更容易懂的话解释章节内容
    - 帮你梳理知识框架
    - 结合代码讲 Agent 的核心概念
    - 帮你总结重点、做题、答疑
    - 设计小练习，带你一步步理解
    
    如果你愿意，可以直接告诉我：
    1. 你正在看第几章/哪一节  
    2. 哪个概念不太懂  
    3. 你想要“通俗解释”还是“结合代码讲解”
    
    你把内容发给我，我们就可以开始。

YOU: 我叫什么名字？
  * summary: [User]: Historical Summary: 对话中，Steven 介绍自己正在学习第 2 部分，助手表示可以用通俗解释、梳理框架、结合代码等方式帮助学习。随后用户问“我叫什么名字？”，对应的名字是 Steven。
    [Assistant]: 你叫 Steven。
AI: 你叫 Steven。

YOU: /quit
